In [1]:
from transformers import BertTokenizer, BertModel
import torch

In [2]:
model_name = "bert-base-uncased"
model = BertModel.from_pretrained(model_name, output_hidden_states=True)
tokenizer = BertTokenizer.from_pretrained(model_name)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
cent_test = "Build, test, and deploy applications using Continuous Integration and Delivery."

In [4]:
token_input = tokenizer(cent_test, return_tensors="pt")

In [5]:
print(token_input)

{'input_ids': tensor([[  101,  3857,  1010,  3231,  1010,  1998, 21296,  5097,  2478,  7142,
          8346,  1998,  6959,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [6]:
model.eval
with torch.no_grad():
    model_output = model(**token_input)

In [7]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [8]:
atten_block_output = model_output[2][1]
atten_block_output

tensor([[[ 0.0340,  0.0078, -0.0754,  ...,  0.2742, -0.1865, -0.0616],
         [ 1.0426, -0.4169, -0.4554,  ..., -0.4859,  0.7546, -0.2820],
         [-0.4079,  0.0518, -0.0017,  ...,  0.3692,  0.7804,  0.2254],
         ...,
         [ 0.4189,  0.2432,  0.6311,  ..., -0.0272, -0.1374, -0.7634],
         [-1.0784,  0.0673,  0.0466,  ...,  0.3830,  0.0236,  0.6729],
         [-0.3686,  0.0117, -0.0812,  ...,  0.1419,  0.0231,  0.1386]]])

In [9]:
embed_output = model.embeddings(token_input["input_ids"], token_input["token_type_ids"])

In [10]:
layer = model.encoder.layer[0]

## 第一次 add & norm，发生在 mha 内部 

In [11]:
mha_output =  layer.attention.self(embed_output)

In [12]:
mha_output

(tensor([[[-0.2457,  0.1456, -0.0182,  ..., -0.0254,  0.0868,  0.0842],
          [-0.5556,  0.1715, -0.0363,  ...,  0.0911,  0.0727, -0.1955],
          [-0.5017,  0.1879,  0.0036,  ..., -0.1120, -0.1093, -0.0741],
          ...,
          [-0.5600,  0.2366, -0.0383,  ..., -0.1218,  0.2498,  0.0089],
          [-0.4819,  0.2408,  0.0031,  ..., -0.0675,  0.1827,  0.0166],
          [-0.2778,  0.2754, -0.0204,  ..., -0.0274,  0.1377,  0.0703]]],
        grad_fn=<ViewBackward0>),)

In [13]:
mha_output[0].shape

torch.Size([1, 15, 768])

In [14]:
attn_output = layer.attention.output(mha_output[0], embed_output)
attn_output.shape

torch.Size([1, 15, 768])

## 第二次 add & norm，发生在 mlp 内部

In [15]:
mlp1 = layer.intermediate(attn_output)
mlp1.shape

torch.Size([1, 15, 3072])

In [16]:
layer.output(mlp1, attn_output)

# 与atten_block_output 相同

tensor([[[ 0.0340,  0.0078, -0.0754,  ...,  0.2742, -0.1865, -0.0616],
         [ 1.0426, -0.4169, -0.4554,  ..., -0.4859,  0.7546, -0.2820],
         [-0.4079,  0.0518, -0.0017,  ...,  0.3692,  0.7804,  0.2254],
         ...,
         [ 0.4189,  0.2432,  0.6311,  ..., -0.0272, -0.1374, -0.7634],
         [-1.0784,  0.0673,  0.0466,  ...,  0.3830,  0.0236,  0.6729],
         [-0.3686,  0.0117, -0.0812,  ...,  0.1419,  0.0231,  0.1386]]],
       grad_fn=<NativeLayerNormBackward0>)